[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P09/P09-2_Etude_Ligne_CORRIGE.ipynb)

# Étude III : tableau de bord d'une ligne

Étude de synthèse de la partie III du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

Cette étude enchaîne ce que les chapitres 7 à 9 ont mis en place : lire un classeur, typer et nettoyer, regrouper, dater, tracer. Un responsable de ligne veut, sur une page, l'état de son parc de machines et l'historique des pannes d'un palier. Deux classeurs fournissent la matière, déjà présentés plus haut : le suivi de production, 20 000 relevés de machines, et le suivi du palier, dont les temps sont comptés depuis le début du suivi. Le résultat attendu est un tableau de bord de trois graphiques, et quatre nombres que l'on peut contrôler : la répartition des machines par état de santé, la vibration moyenne par état, le total des pannes et le mois le plus chargé. La figure du chapitre donne la feuille de route.

Jeu `Suivi_Production_Machines` : `etat_sante_machine`, `indice_sante_machine_0_100`, `niveau_vibrations_en_mm_par_s`.

Jeu `AI4I_maintenance` : `time`, `T_palier_C`, `Panne`.

## Source de données

| Fichier | Lignes | Colonnes | Unités |
|:-|-:|-:|:-|
| `MecaIA-Suivi_Production_Machines.xlsx` | 20000 | 14 | ancienneté en années, complexité et efficacité sans dimension, nombre de capteurs entier, utilisation en proportion, température en °C, vibrations en mm/s, consommation en kWh/h, productivité en pièces/h, indice de santé de 0 à 100 |
| `MecaIA-AI4I_maintenance.xlsx` | 10000 | 9 | temps en jours, températures en °C, vitesse en tr/min, couple en N·m, usure en h, puissance en W, vibration en mm/s |

Les classeurs sont lus dans le dossier `donnees` du dépôt `Meca-PY_Ed2026` ; ouvert seul, par exemple dans Colab, le notebook les télécharge une fois depuis le dépôt public.

### Lecture des jeux

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

In [1]:
from pathlib import Path
import urllib.request

# Les jeux du chapitre : nom du classeur et adresse de la copie publique.
JEUX = {
    "MecaIA-Suivi_Production_Machines.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Suivi_Production_Machines.xlsx",
    "MecaIA-AI4I_maintenance.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-AI4I_maintenance.xlsx",
}

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-PY_Ed2026", "donnees")]
dossier = next((d for d in candidats if all((d / f).is_file() for f in JEUX)), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique des jeux depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    for fichier, adresse in JEUX.items():
        if not (dossier / fichier).is_file():
            try:
                urllib.request.urlretrieve(adresse, dossier / fichier)
            except OSError as erreur:
                raise FileNotFoundError(
                    "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
DONNEES = dossier
print("jeux lus dans le dossier des données :", ", ".join(JEUX))

jeux lus dans le dossier des données : MecaIA-Suivi_Production_Machines.xlsx, MecaIA-AI4I_maintenance.xlsx


### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [2]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("matplotlib", version("matplotlib"))
print("pandas", version("pandas"))

Python 3.12.0
matplotlib 3.11.1
pandas 3.0.5


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook lit les jeux de sa première cellule.

In [3]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = list(JEUX)                                  # les classeurs de la première cellule
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [4]:
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.width", 100)

## Étape 1 : lecture des classeurs

Lecture des classeurs.

In [5]:
# Solution :
data_parc = pd.read_excel(DONNEES / "MecaIA-Suivi_Production_Machines.xlsx")
data_pannes = pd.read_excel(DONNEES / "MecaIA-AI4I_maintenance.xlsx")
parc = data_parc.copy()
pannes = data_pannes.copy()
print("parc :", parc.shape, " pannes :", pannes.shape)
UNITE_TEMPS = {"jours": "D", "heures": "h"}["jours"]
# Fin de la solution

parc : (20000, 14)  pannes : (10000, 9)


## Étape 2 : types et nettoyage

Types et nettoyage. L'état de santé est une catégorie ordonnée du meilleur au pire.

Les contrôles de la cellule lisent la variable `vides` : gardez ce nom.

In [6]:
ordre = ["RAS", "Degradation_legere", "Degradation_severe", "Panne_imminente"]
# Solution :
parc["etat_sante_machine"] = pd.Categorical(parc["etat_sante_machine"], categories=ordre, ordered=True)
vides = int(parc.isna().sum().sum())
colonne_vibration = "niveau_vibrations_en_mm_par_s"
vibration_connue = parc[parc[colonne_vibration].notna()]
print(f"cellules vides : {vides} ; lignes dont la vibration est connue : {len(vibration_connue)}")
# Fin de la solution
assert vides == 6043 and parc["etat_sante_machine"].notna().all(), "aucun état de santé manquant"
print("Étape 2 : contrôles passés")

cellules vides : 6043 ; lignes dont la vibration est connue : 17952
Étape 2 : contrôles passés


## Étape 3 : regroupement par état

Regrouper. Combien de machines par état, et quelle vibration moyenne ?

Les contrôles de la cellule lisent la variable `par_etat` : gardez ce nom.

In [7]:
# Solution :
par_etat = parc.groupby("etat_sante_machine", observed=True).agg(
    machines=("indice_sante_machine_0_100", "count"),
    sante=("indice_sante_machine_0_100", "mean"),
    vibration=(colonne_vibration, "mean"))
part = (par_etat["machines"] / par_etat["machines"].sum() * 100).round(2)
print(par_etat.round(2))
# Fin de la solution
assert (
    par_etat["machines"].sum() == 20000 and par_etat["machines"].tolist() == [11297, 5865, 1382, 1456]
)
assert (
    par_etat["vibration"].is_monotonic_increasing
), "plus l'état est dégradé, plus la vibration moyenne est forte"
assert par_etat["sante"].is_monotonic_decreasing, "l'indice de santé baisse avec la dégradation"
print("Étape 3 : contrôles passés")

                    machines  sante  vibration
etat_sante_machine                            
RAS                    11297  82.15       1.66
Degradation_legere      5865  54.34       2.57
Degradation_severe      1382  37.63       3.09
Panne_imminente         1456  19.38       3.67
Étape 3 : contrôles passés


## Étape 4 : datation des relevés

Dater. Les relevés du palier comptent le temps depuis le début du suivi, dans l'unité de la fiche.

Les contrôles de la cellule lisent les variables `mois_max`, `par_mois` : gardez ces noms.

In [8]:
# Solution :
depart = pd.Timestamp("2017-07-07 08:00")
pannes = pannes.set_index("time")
pannes.index = depart + pd.to_timedelta(pannes.index, unit=UNITE_TEMPS)
pannes.index = pannes.index.round("h")
pannes.index.name = "Horodatage"
par_mois = pannes["Panne"].resample("ME").sum()
mois_max = par_mois.idxmax()
temperature_mensuelle = pannes["T_palier_C"].resample("ME").mean()
print(f"pannes : {int(par_mois.sum())} sur {len(par_mois)} mois ; mois le plus chargé : {mois_max:%Y-%m} avec {int(par_mois.max())}")
# Fin de la solution
assert (
    int(par_mois.sum()) == 339 == int(data_pannes["Panne"].sum())
), "aucune panne perdue par le regroupement"
assert f"{mois_max:%Y-%m}" == "2019-07" and int(par_mois.max()) == 28
print("Étape 4 : contrôles passés")

pannes : 339 sur 55 mois ; mois le plus chargé : 2019-07 avec 28
Étape 4 : contrôles passés


## Étape 5 : tableau de bord

Tableau de bord, trois graphiques lisibles en gris.

Les contrôles de la cellule lisent les variables `fig`, `g2` : gardez ces noms.

In [9]:
# Solution :
fig, (g1, g2, g3) = plt.subplots(1, 3, figsize=(7.0, 2.2))
g1.barh(par_etat.index.astype(str), par_etat["machines"], color="0.55")
g1.set_xlabel("Machines")
g2.plot(par_mois.index, par_mois.values, color="black")
g2.set_ylabel("Pannes par mois")
g3.plot(temperature_mensuelle.index, temperature_mensuelle.values, color="0.4", linestyle="dashed")
g3.set_ylabel("Température moyenne (" + "°C" + ")")
fig.tight_layout()
# Fin de la solution
assert len(fig.axes) == 3 and len(g2.lines) == 1
# Solution :
plt.close(fig)
print("tableau de bord : 3 graphiques")
print("étude III vérifiée")
# Fin de la solution
print("Étape 5 : contrôles passés")

tableau de bord : 3 graphiques
étude III vérifiée
Étape 5 : contrôles passés
